## Test Orbit Determination Functions

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pylupnt as pnt

### Load Ground Station Locations

In [ ]:
from src.odsim import ODSim

et0 = pnt.convert_time(pnt.gregorian_to_time(2025, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)

odconfig = {
    "et0": et0,
    "meas": ["gs"],
    "gs_ids": ["LEGS1_X", "LEGS2_X", "DSS35_X"],
    "init_pos_std": 100e-3,  # km
    "init_vel_std": 1e-3,  # km/s
    "sigma_range": 1e-3,  # m
    "sigma_rangerate": 1e-5,  # m/s
    "elev_mask_deg": 10,  # degrees
    "std_acc": 1e-8,  # km/s^2
}

odsim = ODSim(odconfig)

print("Ground Station Locations (ECEF)")
print(odsim.pos_gs)

print("Ground Station Locations (Lat, Lon, Alt)")
print(odsim.lla_gs)  # lat, lon in degrees, alt in

In [ ]:
from src.postprocess import plot_gs

plot_gs(odsim.lla_gs, odsim.od_config["gs_ids"], markersize=200, fontsize=20)

In [ ]:
M, r = pnt.get_frame_rotation_translation_rv(et0, pnt.MOON_CI, pnt.ECEF)
print("Rotation Matrix from MOON_CI to ECEF")
print(M)
print("Translation Vector from MOON_CI to ECEF")
print(r)

## Dynamics Functions

In [ ]:
# setup satellite positions
from src.constellation_design import setup_walker

ndays = 3
samples_per_hour = 6
tspan = np.linspace(
    0, ndays * 24 * 3600, ndays * 24 * samples_per_hour + 1
)  # 6 samples per hour
et = et0 + tspan  # convert to TAI

# walker parameters
sma = 9000.0
ecc = 0.6
wsign = 1.0
p = 4
t = 2
f = 1
Omega0 = 0.0
x_walker = np.array([sma, ecc, wsign, p, t, f, Omega0])
coes = setup_walker(x_walker)
n_sat = coes.shape[0]
lent = len(tspan)

rv0_mci = np.zeros((n_sat, 6))  # initial states in MOON_CI frame
for si in range(n_sat):
    rv0_op = pnt.classical_to_cart(coes[si, :], pnt.GM_MOON)
    rv0_mci[si] = pnt.convert_frame(tspan[0], rv0_op, pnt.MOON_OP, pnt.MOON_CI)

In [ ]:
dyn = pnt.NBodyDynamics()
dyn.set_integrator(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10))
dyn.add_body(pnt.Body.Moon(20, 20))
dyn.add_body(pnt.Body.Earth())
dyn.add_body(pnt.Body.Sun())
dyn.set_time_step(60)  # 10 seconds time step
dyn.set_frame(pnt.MOON_CI)
dyn.set_autodiff(True)

In [ ]:
from src.odsim import propagate_sats_with_stm

x_sat, stm_sat = propagate_sats_with_stm(rv0_mci, et, dynamics=dyn)

## Measurement Functions

In [ ]:
from src.odsim import get_range_and_rate

elev, y, H = get_range_and_rate(
    tspan,
    odsim.pos_gs,
    x_sat,
    get_H=True,
    add_noise=True,
    sigma_range=1e-3,
    sigma_rangerate=1e-6,
)

In [ ]:
# check elevation angles
n_sat = x_sat.shape[0]
n_gs = odsim.pos_gs.shape[0]
lent = tspan.shape[0]

print("shapes: n_sat =", n_sat, ", n_gs =", n_gs, ", lent =", lent)
print("elev shape:", elev.shape)

elev_mask_deg = 5.0

fig, ax = plt.subplots(n_gs, 1, figsize=(10, 6), sharex=True)
for i in range(n_gs):
    for j in range(n_sat):
        vis_t = elev[i, j, :] >= elev_mask_deg * np.pi / 180.0
        ax[i].plot(
            tspan[vis_t > 0] / 3600,
            np.rad2deg(elev[i, j, vis_t > 0]),
            "o",
            label=f"SAT {j+1}",
        )
    ax[i].set_ylabel(f"GS {i+1} Elevation (deg)")
    ax[i].grid()
    ax[i].legend()
    ax[i].set_ylim(0, 90)
ax[-1].set_xlabel("Time (hours)")
plt.tight_layout()
plt.show()

## OD Simulation

In [ ]:
odconfig = {
    "et0": et0,
    "meas": ["gs"],
    "gs_ids": ["LEGS1_X", "LEGS2_X", "DSS35_X"],
    "init_pos_std": 100e-3,  # km
    "init_vel_std": 1e-3,  # km/s
    "sigma_range": 3e-3,  # m
    "sigma_rangerate": 0.01e-6,  # km/s
    "elev_mask_deg": 10,  # degrees
    "std_acc": 1e-8,  # km/s^2
}

odsim = ODSim(odconfig)

In [ ]:
res = {"x_sat": x_sat, "stm_sat": stm_sat, "elev": elev, "y": y, "H": H}

In [ ]:
from src.odsim import inertial_to_rtn_rotation

R_eci2rtn = inertial_to_rtn_rotation(x_sat[:, :, :3], x_sat[:, :, 3:6])

print("Rotation Matrix from ECI to RTN frame:")
print(R_eci2rtn.shape)

In [ ]:
sigma_sat_mci, sigma_sat_rtn = odsim.run_sim(tspan, rv0_mci, dynamics=dyn, res=res)

In [ ]:
def rms(x, axis=None):
    """Calculate the root mean square of a vector."""
    return np.sqrt(np.mean(x**2, axis=axis))

In [ ]:
fig, ax = plt.subplots(n_sat, 3, figsize=(15, 4 * n_sat), sharex=False)

print("Sat        R [m]          T [m]          N [m]       Norm [m]")
print("---------------------------------------------------------------")
for i in range(n_sat):
    ax[i, 0].plot(tspan / 3600, 1000 * sigma_sat_mci[i, :, 0], label="X")
    ax[i, 0].plot(tspan / 3600, 1000 * sigma_sat_mci[i, :, 1], label="Y")
    ax[i, 0].plot(tspan / 3600, 1000 * sigma_sat_mci[i, :, 2], label="Z")
    ax[i, 0].set_ylabel(f"SAT {i+1} Position (m)")
    ax[i, 0].set_xlabel("Time (hours)")
    ax[i, 0].grid()
    ax[i, 0].set_yscale("log")
    ax[i, 0].legend()

    ax[i, 1].plot(tspan / 3600, 1000 * sigma_sat_rtn[i, :, 0], label="R")
    ax[i, 1].plot(tspan / 3600, 1000 * sigma_sat_rtn[i, :, 1], label="T")
    ax[i, 1].plot(tspan / 3600, 1000 * sigma_sat_rtn[i, :, 2], label="N")
    ax[i, 1].set_ylabel(f"SAT {i+1} RTN Position (m)")
    ax[i, 1].set_xlabel("Time (hours)")
    ax[i, 1].grid()
    ax[i, 1].set_yscale("log")
    ax[i, 1].legend()

    ax[i, 2].plot(
        tspan / 3600, 1000 * np.linalg.norm(sigma_sat_mci[i], axis=1), label="Norm"
    )
    ax[i, 2].set_ylabel(f"SAT {i+1} Norm (m)")
    ax[i, 2].set_xlabel("Time (hours)")
    ax[i, 2].grid()
    ax[i, 2].set_yscale("log")
    ax[i, 2].legend()

    # statistics for last 1/3 of the data
    start_idx = lent // 3 * 2
    pos_rtn_rms = rms(sigma_sat_rtn[i, start_idx:, :], axis=0) * 1000
    pos_rtn_95 = np.percentile(sigma_sat_rtn[i, start_idx:, :], 95, axis=0) * 1000
    pos_norm_rms = rms(np.linalg.norm(sigma_sat_mci[i, start_idx:, :], axis=1)) * 1000
    pos_norm_95 = (
        np.percentile(np.linalg.norm(sigma_sat_mci[i, start_idx:, :], axis=1), 95)
        * 1000
    )
    print(
        f"SAT {i+1}  | "
        f"{pos_rtn_rms[0]:.2f} ({pos_rtn_95[0]:.2f})   "
        f"{pos_rtn_rms[1]:.2f} ({pos_rtn_95[1]:.2f})  "
        f"{pos_rtn_rms[2]:.2f} ({pos_rtn_95[2]:.2f})   "
        f"{pos_norm_rms:.2f} ({pos_norm_95:.2f}) "
    )

plt.tight_layout()
plt.show()